# NB03 — Final Evaluation, Statistics, Figures & Explainability

Produces every number and figure for the Results chapter, and backs them up:
1. **Per-method test performance** (mean ± std over 5 seeds) — pooled and per hospital.
2. **5-seed ensemble** per method with **95% bootstrap CIs** (2,000 resamples).
3. **DeLong tests** for paired AUC comparisons (FL vs local-only, FL vs centralised, FedProx vs FedAvg).
4. **Cross-site generalisation** matrix (a model trained at one hospital, tested at the other).
5. **17 evaluation figures + 9 tables** (ROC, PR, forest plot, seed variability, convergence, per-hospital learning curves, client losses, confusion matrices, per-class AUC, calibration, score distributions, threshold analysis, cross-site heatmaps, attention heatmaps, top-attention tiles; operating points, calibration, communication cost …).
6. Collects the preprocessing (P) and data-split (S) reports from NB01/NB02, so the **results dataset holds every thesis figure and table** with a master index (`THESIS_FIGURES_AND_TABLES.md`).

Every figure is saved as PNG (300 dpi) + PDF; every table as CSV + Markdown + LaTeX. All are pushed to HF and the private Kaggle dataset `panda-fl-v2-phikon-results`.

**Kaggle settings:** CPU is enough (GPU optional) · Internet ON · Add data: `prostate-cancer-grade-assessment` (for heatmaps) · Secrets: `HF_TOKEN` (or `HF_WRITE_TOKEN`), `KAGGLE_KEY` (or `KAGGLE_API_KEY`), `KAGGLE_USERNAME` – tick all three for this notebook.

In [ ]:
import subprocess, sys, os
if os.path.exists('/kaggle/working'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub', 'kaggle',
                    'openslide-python', 'openslide-bin'], check=False)
print('✅ install step done')

In [ ]:
# ── Common infrastructure (identical in all v2 notebooks) ─────────────────────
# Secrets, logging, atomic saves, retries, Hugging Face + Kaggle backup,
# and a session time-guard so work is flushed before Kaggle kills the session.
import os, sys, json, time, glob, shutil, random, subprocess, traceback, hashlib, math, re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import warnings
try:   # small validation sets can miss a grade → sklearn warns on every round; the metric code already handles it (NaN)
    from sklearn.exceptions import UndefinedMetricWarning
    warnings.filterwarnings('ignore', category=UndefinedMetricWarning)
except Exception:
    pass
warnings.filterwarnings('ignore', message='.*Only one class is present in y_true.*')

ON_KAGGLE     = os.path.exists('/kaggle/working')
WORK          = Path('/kaggle/working') if ON_KAGGLE else Path(os.environ.get('FL_WORK', './work')).resolve()
WORK.mkdir(parents=True, exist_ok=True)
SESSION_START = time.time()

KAGGLE_USERNAME_DEFAULT = 'obaidullahmiakhil2'  # used only if no KAGGLE_USERNAME secret exists
# Accepted Kaggle-secret names (first one found wins) – works with either naming style.
SECRET_HF       = ('HF_WRITE_TOKEN', 'HF_TOKEN')
SECRET_KAGGLE   = ('KAGGLE_API_KEY', 'KAGGLE_KEY')
SECRET_KG_USER  = ('KAGGLE_USERNAME',)


def log(msg: str):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | +{(time.time()-SESSION_START)/3600:5.2f}h] {msg}"
    print(line, flush=True)
    try:
        with open(WORK / 'run_log.txt', 'a') as f:
            f.write(line + '\n')
    except Exception:
        pass


def get_secret(names):
    """Return the first available secret among `names` (env var or Kaggle secret)."""
    names = (names,) if isinstance(names, str) else names
    for name in names:
        if os.environ.get(name):
            return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
    except Exception:
        return None
    for name in names:
        try:
            v = client.get_secret(name)
            if v:
                return v.strip()
        except Exception:
            continue
    return None


KAGGLE_USERNAME = (get_secret(SECRET_KG_USER) or KAGGLE_USERNAME_DEFAULT).strip()


def apply_overrides(cfg: dict) -> dict:
    """Allow CFG overrides via env var FL_CFG_OVERRIDE='{"key": value}' (used for testing)."""
    ov = os.environ.get('FL_CFG_OVERRIDE')
    if ov:
        cfg.update(json.loads(ov))
        log(f'CFG overrides applied: {ov}')
    return cfg


def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def atomic_torch_save(obj, path):
    path = Path(path); tmp = path.with_name(path.name + '.tmp')
    torch.save(obj, tmp); os.replace(tmp, path)


def atomic_json(obj, path):
    path = Path(path); tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)


def read_json(path, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return default


def _is_not_found(e):
    """True only for a definite 'file/repo does not exist' answer from the server.
    LocalEntryNotFoundError means 'could not reach the server' → treated as transient."""
    n = type(e).__name__
    if n.startswith('Local') or 'Connection' in n or 'Timeout' in n:
        return False
    return 'NotFound' in n or ' 404' in str(e)[:200]


def retry(fn, tries=5, base_wait=5, what='operation'):
    """Retry transient failures with exponential back-off; 'not found' errors are raised at once."""
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            if _is_not_found(e):
                raise
            log(f'⚠️  {what} failed (attempt {i+1}/{tries}): {type(e).__name__}: {str(e)[:300]}')
            if i == tries - 1:
                raise
            time.sleep(base_wait * 2 ** i)


def hours_elapsed():
    return (time.time() - SESSION_START) / 3600


def time_left_h(max_session_h):
    return max_session_h - hours_elapsed()


class _LocalFakeHub:
    """TEST ONLY: mimics the few HfApi calls we use, backed by a local folder.
    Activated only when env FL_FAKE_HF_DIR is set. Never used on Kaggle."""
    def __init__(self, root):
        self.root = Path(root); self.root.mkdir(parents=True, exist_ok=True)
    def whoami(self): return {'name': 'localtest'}
    def create_repo(self, repo_id, **kw): (self.root / repo_id).mkdir(parents=True, exist_ok=True)
    def upload_file(self, path_or_fileobj, path_in_repo, repo_id, **kw):
        dst = self.root / repo_id / path_in_repo; dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(path_or_fileobj, dst)
    def list_repo_files(self, repo_id, **kw):
        base = self.root / repo_id
        return [str(p.relative_to(base)) for p in base.rglob('*') if p.is_file()]
    def download(self, repo_id, filename, dest):
        src = self.root / repo_id / filename
        if not src.exists(): return None
        dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(src, dest); return dest


class CloudBackup:
    """
    Two independent off-site backups:
      • Hugging Face Hub (private *dataset* repo) – continuous: every file is pushed
        the moment it is written (shards, checkpoints, results).
      • PRIVATE Kaggle Dataset – a new dataset version after EVERY chunk (shard /
        finished experiment) plus a blocking final sync; each version holds all files.
    A failure of one backup never stops training; it is logged and retried.
    """
    def __init__(self, hf_repo_name, kaggle_slug, kaggle_title, local_dir,
                 kaggle_every_h=2.0, enable_hf=True, enable_kaggle=True):
        self.local_dir = Path(local_dir); self.local_dir.mkdir(parents=True, exist_ok=True)
        self.hf_ok, self.kg_ok = False, False
        import threading
        self._kg_lock, self._kg_thread, self._kg_pending = threading.Lock(), None, None
        self._kg_exists, self._kaggle_failed, self.kaggle_versions = False, False, 0
        self.fake = None
        for old in self.local_dir.parent.glob(f'.kg_stage_{self.local_dir.name}_*'):   # leftovers of a killed session
            shutil.rmtree(old, ignore_errors=True)

        # ── Hugging Face ──────────────────────────────────────────────────────
        if os.environ.get('FL_FAKE_HF_DIR'):
            self.fake = _LocalFakeHub(os.environ['FL_FAKE_HF_DIR'])
            self.api, self.token = self.fake, None
            self.hf_repo = f'localtest/{hf_repo_name}'
            self.fake.create_repo(self.hf_repo); self.hf_ok = True
            log(f'🧪 Using LOCAL FAKE HF hub at {self.fake.root}')
        elif enable_hf:
            tok = get_secret(SECRET_HF)
            if tok:
                try:
                    from huggingface_hub import HfApi, login
                    login(token=tok, add_to_git_credential=False)
                    os.environ['HF_TOKEN'] = tok            # authenticated model downloads too
                    self.api, self.token = HfApi(token=tok), tok
                    user = retry(lambda: self.api.whoami()['name'], what='HF whoami')
                    self.hf_repo = f'{user}/{hf_repo_name}'
                    retry(lambda: self.api.create_repo(self.hf_repo, repo_type='dataset',
                                                       private=True, exist_ok=True), what='HF create_repo')
                    self.hf_ok = True
                    log(f'✅ HF backup ON  → https://huggingface.co/datasets/{self.hf_repo}')
                except Exception as e:
                    log(f'❌ HF backup could not start: {e}')
            else:
                log(f'⚠️  HF backup OFF (none of the secrets {SECRET_HF} found – attach one under Add-ons → Secrets)')

        # ── Kaggle ────────────────────────────────────────────────────────────
        if enable_kaggle and (ON_KAGGLE or os.environ.get('FL_TEST_KAGGLE_CLI')):
            key = get_secret(SECRET_KAGGLE)
            if key:
                os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
                os.environ['KAGGLE_KEY'] = key
                if key.startswith('KGAT_'):
                    os.environ['KAGGLE_API_TOKEN'] = key
                self.kg_id = f'{KAGGLE_USERNAME}/{kaggle_slug}'
                atomic_json({'title': kaggle_title[:50], 'id': self.kg_id,
                             'licenses': [{'name': 'CC0-1.0'}]},
                            self.local_dir / 'dataset-metadata.json')
                self.kg_ok = True
                log(f'✅ Kaggle backup ON → https://www.kaggle.com/datasets/{self.kg_id} (private; new version after every chunk)')
            else:
                log(f'⚠️  Kaggle backup OFF (none of the secrets {SECRET_KAGGLE} found)')

    # ── HF helpers ────────────────────────────────────────────────────────────
    def push(self, local_path, path_in_repo=None, msg=None):
        """Upload one file to HF (with retries). Returns True on success."""
        if not self.hf_ok:
            return False
        local_path = Path(local_path)
        path_in_repo = path_in_repo or local_path.name
        try:
            retry(lambda: self.api.upload_file(path_or_fileobj=str(local_path), path_in_repo=path_in_repo,
                                               repo_id=self.hf_repo, repo_type='dataset',
                                               commit_message=msg or f'upload {path_in_repo}'),
                  what=f'HF upload {path_in_repo}')
            return True
        except Exception as e:
            log(f'❌ HF upload permanently failed for {path_in_repo}: {e} (file kept locally; will retry next flush)')
            return False

    def push_many(self, local_paths, msg='batch upload'):
        """Upload several files in ONE commit (keeps HF commit count low)."""
        if not self.hf_ok:
            return False
        local_paths = [Path(p) for p in local_paths if Path(p).exists()]
        if self.fake:
            for p in local_paths:
                self.fake.upload_file(str(p), p.name, self.hf_repo)
            return True
        try:
            from huggingface_hub import CommitOperationAdd
            ops = [CommitOperationAdd(path_in_repo=p.name, path_or_fileobj=str(p)) for p in local_paths]
            retry(lambda: self.api.create_commit(repo_id=self.hf_repo, repo_type='dataset',
                                                 operations=ops, commit_message=msg), what=f'HF commit ({msg})')
            return True
        except Exception as e:
            log(f'❌ HF batch upload failed ({msg}): {e} – files kept locally')
            return False

    def require(self, allow_no_backup=False):
        """Refuse to start long work without the primary (HF) backup – no silent local-only runs."""
        if self.hf_ok or allow_no_backup:
            if not self.kg_ok:
                log('⚠️  Kaggle backup is OFF – HF is still on, continuing.')
            return
        raise RuntimeError('Hugging Face backup is OFF, so progress would only live in this session. '
                           f'Attach a secret named one of {SECRET_HF} (Add-ons → Secrets, tick it for this notebook) '
                           'and re-run. (Set CFG allow_no_backup=True only for throw-away tests.)')

    def remote_files_strict(self):
        """Remote file list, or an exception if HF cannot be reached (never a silent empty list).
        Used before any decision that could overwrite existing cloud data."""
        if not self.hf_ok:
            return None
        return retry(lambda: self.api.list_repo_files(self.hf_repo, repo_type='dataset'), what='HF list (strict)')

    def list_remote(self, prefix=''):
        if not self.hf_ok:
            return []
        try:
            files = retry(lambda: self.api.list_repo_files(self.hf_repo, repo_type='dataset'), what='HF list')
            return [f for f in files if f.startswith(prefix)]
        except Exception:
            return []

    def pull(self, path_in_repo, dest):
        """Download a file from HF into `dest` (full path). Returns Path or None if absent."""
        if not self.hf_ok:
            return None
        dest = Path(dest)
        if self.fake:
            return self.fake.download(self.hf_repo, path_in_repo, dest)
        try:
            from huggingface_hub import hf_hub_download
            p = retry(lambda: hf_hub_download(self.hf_repo, path_in_repo, repo_type='dataset',
                                              token=self.token, local_dir=str(WORK / '_hf_cache')),
                      tries=3, what=f'HF download {path_in_repo}')
            dest.parent.mkdir(parents=True, exist_ok=True)
            tmp = dest.with_name(dest.name + '.tmp')
            shutil.move(str(p), str(tmp)); os.replace(tmp, dest)    # move, not copy: no double disk use
            return dest
        except Exception as e:
            if not _is_not_found(e):
                log(f'   HF download failed: {path_in_repo}: {type(e).__name__}: {str(e)[:150]}')
            return None

    # ── Kaggle helpers ────────────────────────────────────────────────────────
    # Every chunk (feature shard / finished experiment) triggers a new version of the
    # PRIVATE Kaggle dataset. Uploads run in a background thread so the GPU keeps
    # working; chunks finished while an upload is running are coalesced into the
    # next version (each version always contains ALL files written so far).
    def _make_stage(self, msg, stamp):
        """Consistent point-in-time snapshot of local_dir via hard links (no extra disk, no copying).
        Files being written (*.tmp) are skipped, so the training thread is never disturbed."""
        import uuid
        stage = self.local_dir.parent / f'.kg_stage_{self.local_dir.name}_{uuid.uuid4().hex[:6]}'
        stage.mkdir()
        files = []
        for p in sorted(self.local_dir.iterdir()):
            if not p.is_file() or p.name.endswith('.tmp') or p.name == 'kaggle_sync.json':
                continue
            try:
                os.link(p, stage / p.name)
            except FileNotFoundError:
                continue                                       # replaced/removed meanwhile
            except OSError:
                os.symlink(p.resolve(), stage / p.name)        # cross-device fallback
            if p.name != 'dataset-metadata.json':
                files.append(p.name)
        atomic_json({'message': msg, 'time': stamp, 'files': files}, stage / 'kaggle_sync.json')
        return stage, files

    def _kaggle_version_once(self, msg):
        stamp = datetime.now().strftime('%Y-%m-%d %H:%M')
        stage, files = self._make_stage(msg, stamp)
        try:
            return self._kaggle_upload(stage, files, msg, stamp)
        finally:
            shutil.rmtree(stage, ignore_errors=True)

    def _kaggle_upload(self, stage, files, msg, stamp):
        d = str(stage)
        if not self._kg_exists:
            st = subprocess.run(['kaggle', 'datasets', 'status', self.kg_id], capture_output=True, text=True)
            self._kg_exists = st.returncode == 0 and not re.search(r'404|not found|error', st.stdout + st.stderr, re.I)
        # `create` without --public ⇒ the dataset is PRIVATE
        cmd = (['kaggle', 'datasets', 'version', '-p', d, '-m', f'{msg} @ {stamp}'] if self._kg_exists
               else ['kaggle', 'datasets', 'create', '-p', d])
        t0 = time.time()
        r = subprocess.run(cmd, capture_output=True, text=True)
        out = (r.stdout + r.stderr).strip()
        ok = r.returncode == 0 and not re.search(r'\berror\b|failed', out[-400:], re.I)
        if ok:
            self._kg_exists = True
            self.kaggle_versions += 1
        last = out.splitlines()[-1] if out else ''
        log(f'   Kaggle {"version" if cmd[2] == "version" else "CREATE (private)"} '
            f'{"✅" if ok else "❌"} [{msg}] {len(files)} files, {time.time()-t0:.0f}s | {last[:160]}')
        return ok

    def _kaggle_worker(self, msg):
        while True:
            ok = False
            for attempt in range(3):
                try:
                    ok = self._kaggle_version_once(msg)
                except Exception as e:
                    log(f'   Kaggle upload exception: {e}')
                if ok:
                    break
                time.sleep(60 * (attempt + 1))      # e.g. previous version still processing
            if not ok:
                self._kaggle_failed = True          # final sync will retry
            with self._kg_lock:
                if self._kg_pending is None:
                    self._kg_thread = None
                    return
                msg, self._kg_pending = self._kg_pending, None

    def kaggle_push_async(self, msg):
        """Queue a Kaggle dataset version containing everything in local_dir (non-blocking)."""
        if not self.kg_ok:
            return
        import threading
        with self._kg_lock:
            if self._kg_thread is not None:
                self._kg_pending = msg              # coalesce: next version includes this chunk too
                return
            self._kg_thread = threading.Thread(target=self._kaggle_worker, args=(msg,), daemon=True)
            self._kg_thread.start()
        log(f'☁️  Kaggle push queued: {msg}')

    def kaggle_final_sync(self, msg, timeout_min=25):
        """Blocking: wait for any running upload, then make sure the LAST version holds every file."""
        if not self.kg_ok:
            return False
        th = self._kg_thread
        if th is not None:
            log('⏳ waiting for running Kaggle upload to finish …')
            th.join(timeout=timeout_min * 60)
        for attempt in range(4):
            try:
                if self._kaggle_version_once(msg):
                    self._kaggle_failed = False
                    log(f'✅ Kaggle dataset fully synced (private): https://www.kaggle.com/datasets/{self.kg_id}')
                    return True
            except Exception as e:
                log(f'   Kaggle final sync exception: {e}')
            if attempt < 3:
                time.sleep(90 * (attempt + 1))
        log('❌ Kaggle final sync failed – all files are safe on HF and in /kaggle/working; re-run this cell later.')
        return False

    # backwards-compatible name used by older cells
    def kaggle_snapshot(self, msg='checkpoint', force=False):
        return self.kaggle_final_sync(msg) if force else self.kaggle_push_async(msg)


def find_panda_dir():
    cands = ['/kaggle/input/competitions/prostate-cancer-grade-assessment',
             '/kaggle/input/prostate-cancer-grade-assessment',
             os.environ.get('PANDA_DIR', '')]
    for c in cands:
        if c and (Path(c) / 'train.csv').exists():
            return Path(c)
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    for h in hits:
        if (Path(h).parent / 'train_images').exists():
            return Path(h).parent
    return None


ISUP_FROM_GLEASON = {'0+0': 0, 'negative': 0, '3+3': 1, '3+4': 2, '4+3': 3,
                     '4+4': 4, '3+5': 4, '5+3': 4, '4+5': 5, '5+4': 5, '5+5': 5}


def run_names(backbone, smoke=False):
    """All cloud/local names for one run type. Smoke-test runs get a '-smoke' suffix everywhere,
    so they can never mix with (or block) the real run."""
    s, u = ('-smoke', '_smoke') if smoke else ('', '')
    return {'run': 'smoke' if smoke else 'full',
            'features': f'panda-fl-v2-{backbone}-features{s}', 'training': f'panda-fl-v2-{backbone}-training{s}',
            'results': f'panda-fl-v2-{backbone}-results{s}',
            'feat_dir': WORK / f'features_{backbone}{u}', 'out_dir': WORK / f'fl_outputs_{backbone}{u}',
            'fig_dir': WORK / f'results_{backbone}{u}', 'title_sfx': ' SMOKE' if smoke else ''}


def fetch_feature_store(backbone, smoke=False):
    """Locate the NB01 feature store and return (feature_dir, manifest).
    Candidates: this session's /kaggle/working, attached Kaggle input datasets, and HF.
    The copy with the MOST chunks wins (an attached Kaggle version can be older than HF);
    shards already present locally / in /kaggle/input are copied instead of re-downloaded."""
    N = run_names(backbone, smoke)
    repo = N['features']
    best_dir, best_m = None, None
    cands = [N['feat_dir'], Path(str(N['feat_dir']) + '_hf')] + \
            [Path(p).parent for p in glob.glob('/kaggle/input/**/manifest.json', recursive=True)]
    for c in cands:
        m = read_json(Path(c) / 'manifest.json')
        if m and m.get('backbone') == backbone and m.get('run', 'full') == N['run'] \
                and all((Path(c) / s).exists() for s in m['shards']):
            if best_m is None or len(m['shards']) > len(best_m['shards']):
                best_dir, best_m = Path(c), m
    dl = Path(str(N['feat_dir']) + '_hf')
    fb = CloudBackup(repo, repo, repo, dl, enable_kaggle=False)
    hf_m = None
    tmp = WORK / '_hf_manifest_check.json'
    if fb.hf_ok and fb.pull('manifest.json', tmp):
        hf_m = read_json(tmp)
        if hf_m and hf_m.get('run', 'full') != N['run']:
            hf_m = None
    if best_m is not None and (hf_m is None or len(hf_m['shards']) <= len(best_m['shards'])):
        log(f'📦 Feature store: {best_dir} ({len(best_m["shards"])} chunks; HF has {len(hf_m["shards"]) if hf_m else "n/a"})')
        return best_dir, best_m
    assert hf_m is not None, f'Feature store not found locally, in /kaggle/input, or on HF ({repo}). Run NB01 first.'
    dl.mkdir(parents=True, exist_ok=True)
    shutil.copy(tmp, dl / 'manifest.json')
    for i, s in enumerate(hf_m['shards']):
        if (dl / s).exists():
            continue
        if best_dir is not None and (best_dir / s).exists():
            shutil.copy(best_dir / s, dl / s)          # reuse chunk from Kaggle input
        else:
            fb.pull(s, dl / s)
        if i % 10 == 0:
            log(f'   feature chunks ready: {i+1}/{len(hf_m["shards"])}')
    fb.pull('slides_meta.csv', dl / 'slides_meta.csv')
    missing = [s for s in hf_m['shards'] if not (dl / s).exists()]
    assert not missing, f'Could not download chunks: {missing[:5]}'
    log(f'📦 Feature store assembled from HF (+ Kaggle input) at {dl}: {len(hf_m["shards"])} chunks')
    return dl, hf_m


def load_features(feat_dir, manifest, ids=None, with_coords=False):
    """Load shards into RAM: {image_id: fp16 tensor [N, D]} (and coords if asked)."""
    ids = set(ids) if ids is not None else None
    feats, coords = {}, {}
    for s, v in manifest['shards'].items():
        if ids is not None and not ids.intersection(v['ids']):
            continue
        d = torch.load(Path(feat_dir) / s, map_location='cpu', weights_only=True)
        for k, x in d.items():
            if ids is None or k in ids:
                feats[k] = x['feats']
                if with_coords:
                    coords[k] = x['coords']
    return (feats, coords) if with_coords else feats


# ── Thesis figures & tables: one consistent style, PNG(300 dpi)+PDF, CSV+MD+LaTeX ──
# Colours validated for colour-vision deficiency (fixed order, entity-bound, never re-cycled).
METHOD_ORDER  = ['fedprox', 'fedavg', 'centralized', 'local_A_Radboud', 'local_B_Karolinska']
METHOD_COLORS = {'fedprox': '#2a78d6', 'fedavg': '#eb6834', 'centralized': '#1baf7a',
                 'local_A_Radboud': '#eda100', 'local_B_Karolinska': '#e87ba4'}
METHOD_LS     = {'fedprox': '-', 'fedavg': '-', 'centralized': '-', 'local_A_Radboud': '--', 'local_B_Karolinska': '--'}
METHOD_MARK   = {'fedprox': 'o', 'fedavg': 's', 'centralized': 'D', 'local_A_Radboud': '^', 'local_B_Karolinska': 'v'}
METHOD_NICE   = {'centralized': 'Centralised (pooled)', 'local_A_Radboud': 'Local only – Radboud',
                 'local_B_Karolinska': 'Local only – Karolinska', 'fedavg': 'FedAvg', 'fedprox': 'FedProx'}
SITE_COLORS   = {'A_Radboud': '#2a78d6', 'radboud': '#2a78d6', 'B_Karolinska': '#eb6834', 'karolinska': '#eb6834'}
SITE_NICE     = {'A_Radboud': 'Hospital A – Radboud', 'radboud': 'Hospital A – Radboud',
                 'B_Karolinska': 'Hospital B – Karolinska', 'karolinska': 'Hospital B – Karolinska', 'pooled': 'Both hospitals'}
SPLIT_COLORS  = {'train': '#2a78d6', 'val': '#eb6834', 'test': '#1baf7a'}
ISUP_COLORS   = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281', '#0b2a52']   # ordinal ramp, ISUP 0→5
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'


def thesis_style():
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'figure.facecolor': 'white',
        'axes.facecolor': 'white', 'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'axes.titleweight': 'bold',
        'axes.titlesize': 11, 'axes.labelsize': 10, 'axes.spines.top': False, 'axes.spines.right': False,
        'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
        'xtick.color': INK2, 'ytick.color': INK2, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
        'legend.fontsize': 8.5, 'legend.frameon': False, 'lines.linewidth': 2.0, 'lines.markersize': 6,
        'font.family': 'DejaVu Sans', 'text.color': INK, 'pdf.fonttype': 42})
    return plt


class Report:
    """Collects thesis figures/tables for one notebook. Every artefact is written flat into
    `out_dir` (so it lands in the HF repo and the Kaggle dataset) and listed with its caption
    in REPORT_INDEX_<tag>.md / .csv."""
    def __init__(self, out_dir, tag, title):
        self.dir, self.tag, self.title = Path(out_dir), tag, title
        self.dir.mkdir(parents=True, exist_ok=True)
        self.items, self.nf, self.nt = [], 0, 0

    def fig(self, fig, key, caption):
        import matplotlib.pyplot as plt
        self.nf += 1
        base = f'fig_{self.tag}{self.nf:02d}_{key}'
        for ext in ('png', 'pdf'):
            fig.savefig(self.dir / f'{base}.{ext}')
        plt.close(fig)
        self.items.append({'kind': 'figure', 'id': f'Figure {self.tag}{self.nf}', 'file': f'{base}.png',
                           'also': f'{base}.pdf', 'caption': caption})
        log(f'   🖼  {base}.png')
        return base

    def table(self, df, key, caption, index=False, floatfmt=4):
        self.nt += 1
        base = f'tab_{self.tag}{self.nt:02d}_{key}'
        d = df.copy()
        d.to_csv(self.dir / f'{base}.csv', index=index)
        try:
            md = d.round(floatfmt).to_markdown(index=index)
        except Exception:
            md = d.round(floatfmt).to_string(index=index)
        (self.dir / f'{base}.md').write_text(f'**Table {self.tag}{self.nt}.** {caption}\n\n{md}\n')
        try:
            tex = d.round(floatfmt).to_latex(index=index, caption=caption, label=f'tab:{self.tag}{self.nt}', escape=True, float_format=f'%.{floatfmt}f')
            (self.dir / f'{base}.tex').write_text(tex)
        except Exception:
            pass
        self.items.append({'kind': 'table', 'id': f'Table {self.tag}{self.nt}', 'file': f'{base}.csv',
                           'also': f'{base}.md/.tex', 'caption': caption})
        log(f'   📋 {base}.csv')
        return base

    def safe(self, fn, what):
        """Run one figure/table builder; a failure is logged and never stops the notebook."""
        try:
            fn()
        except Exception as e:
            log(f'   ⚠️  {what} skipped: {type(e).__name__}: {e}')

    def finalize(self):
        idx = pd.DataFrame(self.items)
        idx.to_csv(self.dir / f'REPORT_INDEX_{self.tag}.csv', index=False)
        lines = [f'# {self.title}', f'Generated {datetime.now():%Y-%m-%d %H:%M}', '']
        for it in self.items:
            lines.append(f'- **{it["id"]}** — `{it["file"]}` (+ {it["also"]}): {it["caption"]}')
        (self.dir / f'REPORT_INDEX_{self.tag}.md').write_text('\n'.join(lines) + '\n')
        files = [self.dir / f'REPORT_INDEX_{self.tag}.csv', self.dir / f'REPORT_INDEX_{self.tag}.md']
        for it in self.items:
            stem = Path(it['file']).stem
            files += [p for p in self.dir.glob(f'{stem}.*')]
        log(f'📚 {self.title}: {sum(i["kind"]=="figure" for i in self.items)} figures, '
            f'{sum(i["kind"]=="table" for i in self.items)} tables')
        return files


def env_versions():
    """Exact software/hardware versions of this run (saved with the outputs for reproducibility)."""
    import platform, importlib
    v = {'python': platform.python_version(), 'platform': platform.platform(), 'time': str(datetime.now()),
         'on_kaggle': ON_KAGGLE, 'cuda_available': torch.cuda.is_available(),
         'cuda': getattr(torch.version, 'cuda', None),
         'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    for mod in ['torch', 'numpy', 'pandas', 'sklearn', 'scipy', 'transformers', 'huggingface_hub', 'timm',
                'openslide', 'cv2', 'matplotlib']:
        try:
            m = importlib.import_module(mod)
            v[mod] = getattr(m, '__version__', getattr(m, '__library_version__', 'unknown'))
        except Exception:
            v[mod] = None
    return v

In [ ]:
# ── Cell 2: Config + fetch training outputs ─────────────────────────────────
import scipy.stats as st
from sklearn.metrics import (roc_auc_score, roc_curve, cohen_kappa_score, confusion_matrix, precision_recall_curve,
                             average_precision_score, precision_recall_fscore_support, brier_score_loss)
plt = thesis_style(); import matplotlib
import torch.nn as nn

CFG = dict(backbone='phikon', n_boot=2000, boot_seed=7, n_heatmaps=6, heatmap_method='fedprox', heatmap_seed=0,
           # Non-inferiority margins, fixed BEFORE looking at test results (state them in the Methods chapter):
           # FL is 'non-inferior' to a comparator if the lower 95% CI bound of (FL − comparator) > −margin.
           n_boot_diff=1000, ni_margin_auc=0.02, ni_margin_qwk=0.03, allow_no_backup=False,
           smoke=False)   # True = evaluate the NB02 SMOKE run (*-smoke names)
CFG = apply_overrides(CFG)
NAMES = run_names(CFG['backbone'], CFG['smoke'])
if CFG['smoke']:
    CFG.update(n_boot=min(CFG['n_boot'], 200), n_boot_diff=min(CFG['n_boot_diff'], 100))
    log(f'🧪 SMOKE TEST: reading "{NAMES["training"]}", writing "{NAMES["results"]}" (numbers are meaningless on smoke-size data)')
TRAIN_REPO = NAMES['training']
OUT = NAMES['out_dir']
FIG = NAMES['fig_dir']
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

src = CloudBackup(TRAIN_REPO, TRAIN_REPO, TRAIN_REPO, OUT, enable_kaggle=False)
src.require(CFG['allow_no_backup'])
remote = src.list_remote()
for f in remote:
    if (f.startswith(('res__', 'preds__', 'model__')) or f in ('splits.csv', 'config_nb02.json')) and not (OUT / f).exists():
        src.pull(f, OUT / f)
for p in glob.glob(f'/kaggle/input/**/{TRAIN_REPO}/*', recursive=True):   # attached Kaggle copy
    if not (OUT / Path(p).name).exists():
        shutil.copy(p, OUT / Path(p).name)
RES = [read_json(p) for p in sorted(OUT.glob('res__*.json'))]
assert RES, 'No finished experiments found – run NB02 first.'
TCFG = RES[0]['cfg']; HOSP = list(TCFG['hospitals'])
PRED = pd.concat([pd.read_csv(p) for p in sorted(OUT.glob('preds__*.csv'))], ignore_index=True)
METHODS = [m for m in METHOD_ORDER if m in set(PRED.method)]   # fixed colour/order per method
log(f'Loaded {len(RES)} experiment results; methods: {METHODS}; seeds: {sorted(PRED.seed.unique())}')

res_backup = CloudBackup(NAMES['results'], NAMES['results'], f'PANDA FL v2 {CFG["backbone"]} results{NAMES["title_sfx"]}', FIG)
NICE = METHOD_NICE
atomic_json(env_versions(), FIG / 'environment_nb03.json')   # exact versions → HF + Kaggle with the outputs
R = Report(FIG, 'E', 'Evaluation report')
SCOPES = ['pooled'] + HOSP
SC = [f's{k}' for k in range(5)]

In [ ]:
# ── Cell 3: Metric helpers (same definitions as NB02) + DeLong ───────────────
K = 6
def mono_sigmoid(l):
    return np.minimum.accumulate(1 / (1 + np.exp(-np.asarray(l, float))), axis=1)
def class_probs_from_s(s):
    pge = np.concatenate([np.ones((len(s), 1)), s, np.zeros((len(s), 1))], 1)
    p = np.clip(pge[:, :-1] - pge[:, 1:], 0, 1); return p / p.sum(1, keepdims=True)
def grade_from_s(s, thr): return (s > np.asarray(thr)).sum(1)

def metrics_from_s(y, s, thr):
    y = np.asarray(y).astype(int); p = class_probs_from_s(s); g = grade_from_s(s, thr)
    out = {'auc_cancer': roc_auc_score(y >= 1, s[:, 0]), 'auc_cspca': roc_auc_score(y >= 2, s[:, 1])}
    try:
        out['auc_macro_ovr'] = roc_auc_score(y, p, multi_class='ovr', average='macro', labels=list(range(K)))
    except ValueError:
        out['auc_macro_ovr'] = np.nan
    out['qwk'] = cohen_kappa_score(y, g, weights='quadratic'); out['acc'] = float((y == g).mean())
    return out

def tune_thresholds(y, s, passes=3):
    thr = np.full(K - 1, 0.5); grid = np.arange(0.05, 0.951, 0.025)
    best = cohen_kappa_score(y, grade_from_s(s, thr), weights='quadratic')
    for _ in range(passes):
        for k in range(K - 1):
            for t in grid:
                c = thr.copy(); c[k] = t
                q = cohen_kappa_score(y, grade_from_s(s, c), weights='quadratic')
                if q > best + 1e-9: best, thr = q, c
    return thr

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    T2 = np.empty(N); T2[J] = T; return T2

def delong_test(y, s1, s2):
    """Paired DeLong test (DeLong 1988; fast version Sun & Xu 2014). Returns auc1, auc2, p-value."""
    y = np.asarray(y).astype(bool); pos, neg = np.where(y)[0], np.where(~y)[0]; m, n = len(pos), len(neg)
    aucs, V10, V01 = [], [], []
    for s in (s1, s2):
        s = np.asarray(s, float); tx, ty, tz = _midrank(s[pos]), _midrank(s[neg]), _midrank(np.r_[s[pos], s[neg]])
        auc = tz[:m].sum() / m / n - (m + 1) / (2 * n); aucs.append(auc)
        V10.append((tz[:m] - tx) / n); V01.append(1 - (tz[m:] - ty) / m)
    S10, S01 = np.cov(np.array(V10)), np.cov(np.array(V01))
    S = S10 / m + S01 / n; var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(max(var, 1e-12))
    return aucs[0], aucs[1], float(2 * st.norm.sf(abs(z)))

def bootstrap_ci(y, s, thr, n=CFG['n_boot'], seed=CFG['boot_seed']):
    rng = np.random.default_rng(seed); y = np.asarray(y); vals = []
    need = K if len(np.unique(y)) == K else 2          # tiny test sets (smoke run) may lack some grades
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < need or len(np.unique(y[i] >= 1)) < 2 or len(np.unique(y[i] >= 2)) < 2: continue
        try:
            vals.append(metrics_from_s(y[i], s[i], thr))
        except ValueError:
            continue
    df = pd.DataFrame(vals); return df.quantile(0.025), df.quantile(0.975)

def ece(y_bin, p, bins=10):
    y_bin, p = np.asarray(y_bin).astype(float), np.asarray(p, float); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for a, b in zip(edges[:-1], edges[1:]):
        m = (p >= a) & (p < b) if b < 1 else (p >= a) & (p <= b)
        if m.any(): e += m.mean() * abs(y_bin[m].mean() - p[m].mean())
    return float(e)

def holm(p):
    """Holm–Bonferroni step-down adjusted p-values (controls family-wise error rate)."""
    p = np.asarray(p, float); m = len(p); o = np.argsort(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj

def sel(df, split, scope):
    d = df[df.split == split]
    return d if scope == 'pooled' else d[d.hospital == scope]

In [ ]:
# ── Cell 4: Seed ensembles (probabilities averaged over seeds; thresholds tuned on validation) ─
ENS = {}
for m in METHODS:
    d = PRED[PRED.method == m]
    S = np.stack([mono_sigmoid(g.sort_values('image_id')[[f'logit{k}' for k in range(K - 1)]].values) for _, g in d.groupby('seed')])
    base = d[d.seed == d.seed.min()].sort_values('image_id')[['image_id', 'hospital', 'isup_grade', 'split']].reset_index(drop=True)
    base[SC] = S.mean(0)
    v = base[base.split == 'val']
    ENS[m] = (base, tune_thresholds(v.isup_grade.values, v[SC].values))
best_fl = max([m for m in METHODS if m in ('fedavg', 'fedprox')] or METHODS,
              key=lambda m: metrics_from_s(sel(ENS[m][0], 'val', 'pooled').isup_grade.values, sel(ENS[m][0], 'val', 'pooled')[SC].values, ENS[m][1])['qwk'])
log(f'Seeds per method: {PRED.groupby("method").seed.nunique().to_dict()} | best FL (validation QWK): {best_fl}')

In [ ]:
# ── Cell 5: TABLES ───────────────────────────────────────────────────────────
CI = {}   # (method, scope) -> dict(metric -> (pt, lo, hi)) reused by the forest plot

def t_mean_std():
    rows = [{'method': r['method'], 'seed': r['seed'], 'scope': sc, **mm['thr_tuned']} for r in RES for sc, mm in r['test'].items()]
    T = pd.DataFrame(rows); cols = ['auc_cancer', 'auc_cspca', 'auc_macro_ovr', 'qwk', 'acc', 'bal_acc']
    g = T.groupby(['scope', 'method'])[cols].agg(['mean', 'std'])
    out = pd.DataFrame({c: g[(c, 'mean')].map('{:.4f}'.format) + ' ± ' + g[(c, 'std')].map('{:.4f}'.format) for c in cols})
    out = out.reindex([(s_, m) for s_ in SCOPES for m in METHODS if (s_, m) in out.index]).reset_index()
    out['scope'] = out['scope'].map(SITE_NICE); out['method'] = out['method'].map(NICE)
    R.table(out, 'test_mean_std', 'Test performance per method (mean ± SD over seeds; ordinal thresholds tuned on validation). AUC cancer = ISUP ≥ 1 vs 0; AUC csPCa = ISUP ≥ 2 vs < 2; macro AUC = 6-class one-vs-rest.')
    raw = T.copy(); raw['method'] = raw['method'].map(NICE)
    R.table(raw.sort_values(['scope', 'method', 'seed']), 'test_per_seed_raw', 'Raw per-seed test metrics for every method and scope (appendix).')

def t_ensemble_ci():
    rows = []
    for m in METHODS:
        b, thr = ENS[m]
        for sc in SCOPES:
            t = sel(b, 'test', sc); y, s_ = t.isup_grade.values, t[SC].values
            pt = metrics_from_s(y, s_, thr); lo, hi = bootstrap_ci(y, s_, thr)
            CI[(m, sc)] = {c: (pt[c], lo[c], hi[c]) for c in ['auc_cancer', 'auc_cspca', 'auc_macro_ovr', 'qwk']}
            rows.append({'Method': NICE[m], 'Test set': SITE_NICE[sc], 'n': len(y),
                         **{c: f'{pt[c]:.4f} [{lo[c]:.4f}, {hi[c]:.4f}]' for c in ['auc_cancer', 'auc_cspca', 'auc_macro_ovr', 'qwk']}})
    R.table(pd.DataFrame(rows), 'ensemble_95ci', f'Seed-ensemble test performance with 95% bootstrap confidence intervals ({CFG["n_boot"]} resamples).')

def t_delong():
    pairs = [('fedprox', 'fedavg'), ('fedavg', 'local_A_Radboud'), ('fedavg', 'local_B_Karolinska'), ('fedprox', 'local_A_Radboud'),
             ('fedprox', 'local_B_Karolinska'), ('fedavg', 'centralized'), ('fedprox', 'centralized')]
    rows = []
    for a, b in pairs:
        if a not in ENS or b not in ENS: continue
        for sc in SCOPES:
            ta, tb = sel(ENS[a][0], 'test', sc).sort_values('image_id'), sel(ENS[b][0], 'test', sc).sort_values('image_id')
            for task, k, cut in [('Cancer (ISUP ≥ 1)', 0, 1), ('csPCa (ISUP ≥ 2)', 1, 2)]:
                a1, a2, pv = delong_test(ta.isup_grade.values >= cut, ta[f's{k}'].values, tb[f's{k}'].values)
                rows.append({'A': NICE[a], 'B': NICE[b], 'Test set': SITE_NICE[sc], 'Task': task, 'AUC A': a1, 'AUC B': a2,
                             'Δ (A−B)': a1 - a2, 'p (DeLong)': pv})
    T = pd.DataFrame(rows)
    T['p (Holm-adjusted)'] = holm(T['p (DeLong)'].values)
    T['Significant (Holm, α=0.05)'] = np.where(T['p (Holm-adjusted)'] < 0.05, 'yes', 'no')
    R.table(T, 'delong_tests', f'Paired DeLong tests comparing ROC-AUCs of seed-ensemble models on the same test slides; '
            f'p-values adjusted for {len(T)} comparisons with the Holm–Bonferroni method.')

def t_paired_differences():
    """Paired bootstrap (same resampled slides for both models) of metric differences on the pooled test set."""
    comps = [(f, c) for f in ('fedprox', 'fedavg') for c in ('centralized', 'local_A_Radboud', 'local_B_Karolinska') if f in ENS and c in ENS]
    rng = np.random.default_rng(CFG['boot_seed']); rows = []
    base = sel(ENS[METHODS[0]][0], 'test', 'pooled').sort_values('image_id'); y = base.isup_grade.values; n = len(y)
    need = K if len(np.unique(y)) == K else 2
    idx = [i for i in (rng.integers(0, n, n) for _ in range(CFG['n_boot_diff']))
           if len(np.unique(y[i])) >= need and len(np.unique(y[i] >= 1)) == 2 and len(np.unique(y[i] >= 2)) == 2]
    for f, c in comps:
        tf, tc = [sel(ENS[m][0], 'test', 'pooled').sort_values('image_id') for m in (f, c)]
        assert (tf.image_id.values == base.image_id.values).all() and (tc.image_id.values == base.image_id.values).all()
        sf, sc_ = tf[SC].values, tc[SC].values; thf, thc = ENS[f][1], ENS[c][1]
        pt_f, pt_c = metrics_from_s(y, sf, thf), metrics_from_s(y, sc_, thc)
        D = pd.DataFrame([{k: metrics_from_s(y[i], sf[i], thf)[k] - metrics_from_s(y[i], sc_[i], thc)[k]
                           for k in ('auc_cancer', 'auc_cspca', 'auc_macro_ovr', 'qwk')} for i in idx])
        for k, lab in [('auc_cancer', 'AUC cancer'), ('auc_cspca', 'AUC csPCa'), ('auc_macro_ovr', 'Macro AUC'), ('qwk', 'QWK')]:
            lo, hi = D[k].quantile(0.025), D[k].quantile(0.975); mg = CFG['ni_margin_qwk'] if k == 'qwk' else CFG['ni_margin_auc']
            rows.append({'FL method': NICE[f], 'Comparator': NICE[c], 'Metric': lab, 'Δ (FL − comparator)': pt_f[k] - pt_c[k],
                         '95% CI low': lo, '95% CI high': hi, 'Margin': mg,
                         'Non-inferior (CI low > −margin)': 'yes' if lo > -mg else 'no',
                         'Superior (CI low > 0)': 'yes' if lo > 0 else 'no'})
    R.table(pd.DataFrame(rows), 'paired_differences_noninferiority',
            f'Paired bootstrap ({len(idx)} resamples of the pooled test set) of metric differences between each FL method and each comparator; '
            f'non-inferiority margins ({CFG["ni_margin_auc"]} AUC, {CFG["ni_margin_qwk"]} QWK) were fixed before evaluation.')

def t_cross_site():
    rows = []
    for m in METHODS:
        b, thr = ENS[m]
        for h in HOSP:
            t = sel(b, 'test', h); mm = metrics_from_s(t.isup_grade.values, t[SC].values, thr)
            rows.append({'Method (trained on)': NICE[m], 'Tested at': SITE_NICE[h], **{k: round(v, 4) for k, v in mm.items()}})
    R.table(pd.DataFrame(rows), 'cross_site', 'Cross-site generalisation: every model evaluated separately on each hospital\'s test set.')

def t_per_class():
    rows = []
    for m in [best_fl, 'centralized']:
        if m not in ENS: continue
        b, thr = ENS[m]; t = sel(b, 'test', 'pooled'); y = t.isup_grade.values; s_ = t[SC].values
        g = grade_from_s(s_, thr); p = class_probs_from_s(s_)
        pr, rc, f1, sup = precision_recall_fscore_support(y, g, labels=range(K), zero_division=0)
        for c in range(K):
            rows.append({'Method': NICE[m], 'ISUP': c, 'Support': int(sup[c]), 'Precision': pr[c], 'Recall': rc[c], 'F1': f1[c],
                         'One-vs-rest AUC': roc_auc_score(y == c, p[:, c]) if 0 < (y == c).sum() < len(y) else np.nan})
    R.table(pd.DataFrame(rows), 'per_class_metrics', 'Per-ISUP-grade precision, recall, F1 and one-vs-rest AUC on the pooled test set (best FL method and centralised).')

OPS = {}
def t_operating_points():
    rows = []
    for m in METHODS:
        b, _ = ENS[m]; v, t = sel(b, 'val', 'pooled'), sel(b, 'test', 'pooled')
        for task, k, cut in [('Cancer (ISUP ≥ 1)', 0, 1), ('csPCa (ISUP ≥ 2)', 1, 2)]:
            fpr, tpr, th = roc_curve(v.isup_grade >= cut, v[f's{k}'])
            cands = {'Youden (val)': th[np.argmax(tpr - fpr)],
                     'Sensitivity ≥ 90% (val)': th[np.argmax(tpr >= 0.90)], 'Sensitivity ≥ 95% (val)': th[np.argmax(tpr >= 0.95)]}
            for name, thv in cands.items():
                yt = (t.isup_grade >= cut).values; pt = (t[f's{k}'] >= thv).values
                tp, fn, fp, tn = (yt & pt).sum(), (yt & ~pt).sum(), (~yt & pt).sum(), (~yt & ~pt).sum()
                OPS[(m, k, name)] = thv
                rows.append({'Method': NICE[m], 'Task': task, 'Operating point': name, 'Threshold': thv,
                             'Sensitivity': tp / max(tp + fn, 1), 'Specificity': tn / max(tn + fp, 1),
                             'PPV': tp / max(tp + fp, 1), 'NPV': tn / max(tn + fn, 1), 'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn})
    R.table(pd.DataFrame(rows), 'operating_points', 'Clinical operating points: thresholds chosen on validation, applied unchanged to the pooled test set.')

def t_calibration():
    rows = []
    for m in METHODS:
        t = sel(ENS[m][0], 'test', 'pooled')
        for task, k, cut in [('Cancer (ISUP ≥ 1)', 0, 1), ('csPCa (ISUP ≥ 2)', 1, 2)]:
            yb = (t.isup_grade >= cut).values
            rows.append({'Method': NICE[m], 'Task': task, 'Brier score': brier_score_loss(yb, t[f's{k}']), 'ECE (10 bins)': ece(yb, t[f's{k}'])})
    R.table(pd.DataFrame(rows), 'calibration', 'Probability calibration on the pooled test set (lower is better).')

def t_cost():
    rows = []
    for m in METHODS:
        rr = [r for r in RES if r['method'] == m]
        mp = OUT / f'model__{m}__s{rr[0]["seed"]}.pt'
        n_par = sum(v.numel() for v in torch.load(mp, map_location='cpu', weights_only=False)['state'].values()) if mp.exists() else np.nan
        n_cli = len(rr[0]['n_train']); fl = m in ('fedavg', 'fedprox')
        per_round = n_par * 4 * 2 * n_cli / 1e6 if fl else 0.0
        br = np.array([r['best_round'] for r in rr]); secs = np.array([np.mean([h['sec'] for h in r['history']]) for r in rr])
        rows.append({'Method': NICE[m], 'Clients': n_cli, 'Training slides': sum(rr[0]['n_train'].values()), 'Model parameters': n_par,
                     'Rounds / epochs': TCFG['rounds'], 'Best round (mean ± SD)': f'{br.mean():.1f} ± {br.std():.1f}',
                     'Seconds per round': round(float(secs.mean()), 2),
                     'Communication per round (MB, all clients, up+down)': round(per_round, 3),
                     'Total communication (MB)': round(per_round * TCFG['rounds'], 2),
                     'Raw images shared': 'none' if (fl or m.startswith('local')) else 'all (pooled)'})
    R.table(pd.DataFrame(rows), 'training_and_communication_cost', 'Training cost and communication overhead; FL exchanges only model weights (float32), never images or features.')

for fn, what in [(t_mean_std, 'mean±std table'), (t_ensemble_ci, 'ensemble CI table'), (t_delong, 'DeLong table'),
                 (t_paired_differences, 'paired differences / non-inferiority'), (t_cross_site, 'cross-site table'), (t_per_class, 'per-class table'), (t_operating_points, 'operating points'),
                 (t_calibration, 'calibration table'), (t_cost, 'cost table')]:
    R.safe(fn, what)

In [ ]:
# ── Cell 6: FIGURES – discrimination (ROC, PR, forest, seeds, headline bars) ──
def legend_label(m, auc): return f'{NICE[m]} (AUC {auc:.3f})'

def f_roc(k, cut, task):
    fig, axes = plt.subplots(1, len(SCOPES), figsize=(5.2 * len(SCOPES), 4.8), sharey=True)
    for ax, sc in zip(axes, SCOPES):
        for m in METHODS:
            t = sel(ENS[m][0], 'test', sc); fpr, tpr, _ = roc_curve(t.isup_grade >= cut, t[f's{k}'])
            ax.plot(fpr, tpr, color=METHOD_COLORS[m], ls=METHOD_LS[m], label=legend_label(m, roc_auc_score(t.isup_grade >= cut, t[f's{k}'])))
        ax.plot([0, 1], [0, 1], color=GRID, lw=1, ls=':'); ax.set_title(SITE_NICE[sc]); ax.set_xlabel('1 − specificity'); ax.legend(loc='lower right', fontsize=7.5)
    axes[0].set_ylabel('Sensitivity'); fig.suptitle(f'ROC curves – {task} (test set, seed ensembles)', fontweight='bold', y=1.02)
    R.fig(fig, f'roc_isup_ge{cut}', f'ROC curves for {task} on the held-out test set (pooled and per hospital). Solid = federated/centralised, dashed = single-hospital models.')

def f_pr():
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, (k, cut, task) in zip(axes, [(0, 1, 'Cancer (ISUP ≥ 1)'), (1, 2, 'csPCa (ISUP ≥ 2)')]):
        for m in METHODS:
            t = sel(ENS[m][0], 'test', 'pooled'); pr, rc, _ = precision_recall_curve(t.isup_grade >= cut, t[f's{k}'])
            ax.plot(rc, pr, color=METHOD_COLORS[m], ls=METHOD_LS[m], label=f'{NICE[m]} (AP {average_precision_score(t.isup_grade >= cut, t[f"s{k}"]):.3f})')
        prev = (sel(ENS[METHODS[0]][0], 'test', 'pooled').isup_grade >= cut).mean()
        ax.axhline(prev, color=INK2, ls=':', lw=1); ax.text(0.02, prev + 0.01, f'prevalence {prev:.2f}', fontsize=7.5, color=INK2)
        ax.set_title(task); ax.set_xlabel('Recall (sensitivity)'); ax.set_ylabel('Precision (PPV)'); ax.legend(loc='lower left', fontsize=7.5)
    fig.suptitle('Precision–recall curves (pooled test set)', fontweight='bold', y=1.02)
    R.fig(fig, 'precision_recall', 'Precision–recall curves with average precision (AP) on the pooled test set; dotted line = class prevalence (chance level).')

def f_forest():
    mets = [('auc_cancer', 'AUC cancer'), ('auc_cspca', 'AUC csPCa'), ('auc_macro_ovr', 'Macro AUC (6-class)'), ('qwk', 'QWK')]
    fig, axes = plt.subplots(1, 4, figsize=(15, 0.55 * len(METHODS) * len(SCOPES) + 1.5), sharey=True)
    ylab, y = [], 0
    for sc in SCOPES:
        for m in METHODS:
            for ax, (c, _) in zip(axes, mets):
                pt, lo, hi = CI[(m, sc)][c]
                ax.errorbar(pt, y, xerr=[[pt - lo], [hi - pt]], fmt=METHOD_MARK[m], color=METHOD_COLORS[m], ms=7, capsize=3, lw=1.5)
            ylab.append(f'{NICE[m]}  ·  {sc.split("_")[-1] if sc != "pooled" else "both"}'); y += 1
        y += 0.6
    ys = [i + 0.6 * (i // len(METHODS)) for i in range(len(ylab))]
    axes[0].set_yticks(ys); axes[0].set_yticklabels(ylab, fontsize=8); axes[0].invert_yaxis()
    for ax, (_, t) in zip(axes, mets): ax.set_title(t); ax.grid(axis='y', visible=False)
    fig.suptitle('Test performance with 95% bootstrap confidence intervals', fontweight='bold', y=1.0)
    R.fig(fig, 'forest_plot_ci', 'Forest plot of seed-ensemble test metrics with 95% bootstrap CIs, per method and test hospital.')

def f_headline():
    mets = [('auc_cancer', 'AUC\ncancer'), ('auc_cspca', 'AUC\ncsPCa'), ('auc_macro_ovr', 'Macro AUC\n(6-class)'), ('qwk', 'QWK')]
    fig, ax = plt.subplots(figsize=(10, 4.4)); w = 0.8 / len(METHODS)
    for i, m in enumerate(METHODS):
        pts = [CI[(m, 'pooled')][c] for c, _ in mets]
        x = np.arange(len(mets)) + (i - (len(METHODS) - 1) / 2) * w
        ax.bar(x, [p[0] for p in pts], w, color=METHOD_COLORS[m], edgecolor='white', linewidth=1.5, label=NICE[m],
               yerr=[[p[0] - p[1] for p in pts], [p[2] - p[0] for p in pts]], error_kw=dict(ecolor=INK2, lw=1, capsize=2))
    lo = min(CI[(m, 'pooled')][c][1] for m in METHODS for c, _ in mets)
    ax.set_ylim(max(0, lo - 0.05), 1.0); ax.set_xticks(range(len(mets))); ax.set_xticklabels([t for _, t in mets])
    ax.set_ylabel('Score (pooled test)'); ax.set_title('Headline comparison of all methods'); ax.legend(ncol=3, fontsize=8, loc='lower center', bbox_to_anchor=(.5, 1.08))
    ax.grid(axis='x', visible=False)
    R.fig(fig, 'headline_metrics', 'Headline metrics on the pooled test set for every method (bars = seed ensemble, whiskers = 95% bootstrap CI). Y-axis starts above zero to show differences.')

def f_seed_var():
    rows = [{'method': r['method'], 'seed': r['seed'], **r['test']['pooled']['thr_tuned']} for r in RES]
    T = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (c, t) in zip(axes, [('auc_cspca', 'AUC csPCa'), ('auc_macro_ovr', 'Macro AUC'), ('qwk', 'QWK')]):
        for i, m in enumerate(METHODS):
            v = T[T.method == m][c].values
            ax.scatter(np.full(len(v), i) + np.linspace(-.12, .12, len(v)), v, color=METHOD_COLORS[m], marker=METHOD_MARK[m], s=40, zorder=3)
            ax.hlines(v.mean(), i - .25, i + .25, color=INK, lw=2)
        ax.set_xticks(range(len(METHODS))); ax.set_xticklabels([NICE[m].replace(' – ', '\n') for m in METHODS], fontsize=7.5); ax.set_title(t)
    axes[0].set_ylabel('Pooled test score per seed')
    R.fig(fig, 'seed_variability', 'Run-to-run variability: each dot is one training seed, the black bar is the mean.')

for fn, what in [(lambda: f_roc(0, 1, 'cancer vs benign (ISUP ≥ 1)'), 'ROC cancer'), (lambda: f_roc(1, 2, 'clinically significant cancer (ISUP ≥ 2)'), 'ROC csPCa'),
                 (f_pr, 'PR curves'), (f_forest, 'forest plot'), (f_headline, 'headline bars'), (f_seed_var, 'seed variability')]:
    R.safe(fn, what)

In [ ]:
# ── Cell 7: FIGURES – training dynamics (convergence, per-hospital curves, client losses) ─
def hist_df():
    rows = []
    for r in RES:
        for h in r['history']:
            base = {'method': r['method'], 'seed': r['seed'], 'round': h['round'], 'val_score': h['val_score'], 'lr': h['lr']}
            for c, l in h['train_loss'].items(): rows.append({**base, 'kind': 'loss', 'who': c, 'value': l})
            for hh, mm in h['val'].items():
                for met in ('auc_cspca', 'auc_cancer', 'qwk', 'auc_macro_ovr'): rows.append({**base, 'kind': met, 'who': hh, 'value': mm[met]})
    return pd.DataFrame(rows)
H = hist_df()

def band(ax, d, color, ls, label):
    g = d.groupby('round')['value'].agg(['mean', 'std']).fillna(0)
    ax.plot(g.index, g['mean'], color=color, ls=ls, label=label); ax.fill_between(g.index, g['mean'] - g['std'], g['mean'] + g['std'], color=color, alpha=.15, lw=0)

def f_convergence():
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    for m in METHODS:
        d = H[(H.method == m) & (H.kind == 'loss')].drop_duplicates(['seed', 'round'])[['round', 'val_score']].rename(columns={'val_score': 'value'})
        band(ax, d, METHOD_COLORS[m], METHOD_LS[m], NICE[m])
    ax.set_xlabel('Communication round (FL) / epoch'); ax.set_ylabel('Validation score = mean(QWK, macro AUC)')
    ax.set_title('Convergence on validation data (mean ± SD over seeds)'); ax.legend(fontsize=8)
    R.fig(fig, 'convergence_val_score', 'Validation model-selection score per round (mean ± SD over seeds). FL rounds and centralised epochs use the same data budget.')

def f_per_hospital_curves():
    for met, lab in [('auc_cspca', 'Validation AUC csPCa'), ('qwk', 'Validation QWK')]:
        fig, axes = plt.subplots(1, len(HOSP), figsize=(12, 4.2), sharey=True)
        for ax, h in zip(axes, HOSP):
            for m in METHODS:
                d = H[(H.method == m) & (H.kind == met) & (H.who == h)]
                if len(d): band(ax, d, METHOD_COLORS[m], METHOD_LS[m], NICE[m])
            ax.set_title(f'Validated at {SITE_NICE[h]}'); ax.set_xlabel('Round / epoch')
        axes[0].set_ylabel(lab); axes[0].legend(fontsize=7.5)
        R.fig(fig, f'per_hospital_val_{met}', f'{lab} measured separately at each hospital during training (mean ± SD over seeds). Local models are validated only at their own hospital.')

def f_client_loss():
    fig, ax = plt.subplots(figsize=(8.5, 4.3))
    for m in [x for x in METHODS if x in ('fedavg', 'fedprox', 'centralized')]:
        for who in sorted(H[(H.method == m) & (H.kind == 'loss')].who.unique()):
            d = H[(H.method == m) & (H.kind == 'loss') & (H.who == who)]
            band(ax, d, METHOD_COLORS[m], '-' if who != HOSP[-1] else ':', f'{NICE[m]} – {SITE_NICE.get(who, "pooled data") if who != "pooled" else "pooled data"}')
    ax.set_xlabel('Round / epoch'); ax.set_ylabel('Training loss (ordinal BCE)'); ax.set_title('Local training loss per client'); ax.legend(fontsize=7.5)
    R.fig(fig, 'client_training_loss', 'Training loss of each federated client per round (solid = Radboud, dotted = Karolinska) compared with centralised training.')

def f_lr():
    d = H[(H.kind == 'loss')].drop_duplicates(['round'])[['round', 'lr']].sort_values('round')
    fig, ax = plt.subplots(figsize=(6.5, 3)); ax.plot(d['round'], d['lr'], color='#2a78d6', marker='o', ms=3)
    ax.set_xlabel('Round / epoch'); ax.set_ylabel('Learning rate'); ax.set_title('Learning-rate schedule (warm-up + cosine)')
    R.fig(fig, 'lr_schedule', 'Learning-rate schedule shared by all methods (linear warm-up followed by cosine decay).')

for fn, what in [(f_convergence, 'convergence'), (f_per_hospital_curves, 'per-hospital curves'), (f_client_loss, 'client loss'), (f_lr, 'LR schedule')]:
    R.safe(fn, what)

In [ ]:
# ── Cell 8: FIGURES – error analysis (confusion, per-class, calibration, scores, thresholds, cross-site) ─
def f_confusion_grid():
    n = len(METHODS); fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.8))
    for ax, m in zip(np.atleast_1d(axes), METHODS):
        b, thr = ENS[m]; t = sel(b, 'test', 'pooled')
        cm = confusion_matrix(t.isup_grade, grade_from_s(t[SC].values, thr), labels=range(K)).astype(float)
        cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(cmn, cmap=matplotlib.colors.LinearSegmentedColormap.from_list('b', ['#ffffff'] + ISUP_COLORS), vmin=0, vmax=1)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if cmn[i, j] > .5 else INK)
        ax.set_title(NICE[m], fontsize=9); ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.grid(False); ax.set_xlabel('Predicted ISUP')
    np.atleast_1d(axes)[0].set_ylabel('True ISUP')
    R.fig(fig, 'confusion_matrices_all', 'Row-normalised confusion matrices on the pooled test set for every method (diagonal = correct grade).')
    b, thr = ENS[best_fl]; t = sel(b, 'test', 'pooled'); cm = confusion_matrix(t.isup_grade, grade_from_s(t[SC].values, thr), labels=range(K))
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(cm, cmap=matplotlib.colors.LinearSegmentedColormap.from_list('b', ['#ffffff'] + ISUP_COLORS))
    for i in range(K):
        for j in range(K):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=9, color='white' if cm[i, j] > cm.max() / 2 else INK)
    ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.grid(False); ax.set_xlabel('Predicted ISUP'); ax.set_ylabel('True ISUP')
    ax.set_title(f'{NICE[best_fl]} – counts')
    R.fig(fig, 'confusion_best_fl_counts', f'Confusion matrix (counts) of the best federated model ({NICE[best_fl]}, chosen on validation) on the pooled test set.')

def f_per_class_auc():
    fig, ax = plt.subplots(figsize=(9, 4)); w = 0.8 / len(METHODS); lo_ = 0.5
    for i, m in enumerate(METHODS):
        t = sel(ENS[m][0], 'test', 'pooled'); y = t.isup_grade.values; p = class_probs_from_s(t[SC].values)
        a = [roc_auc_score(y == c, p[:, c]) if 0 < (y == c).sum() < len(y) else np.nan for c in range(K)]
        ax.bar(np.arange(K) + (i - (len(METHODS) - 1) / 2) * w, a, w, color=METHOD_COLORS[m], edgecolor='white', linewidth=1.2, label=NICE[m])
        lo_ = min(lo_, np.nanmin(a) - 0.05)
    ax.axhline(0.5, color=INK2, ls=':', lw=1)
    ax.set_xticks(range(K)); ax.set_xticklabels([f'ISUP {c}' for c in range(K)]); ax.set_ylim(max(0, lo_), 1.0)
    ax.set_ylabel('One-vs-rest AUC'); ax.set_title('Discrimination per ISUP grade'); ax.legend(ncol=3, fontsize=7.5, loc='lower center', bbox_to_anchor=(.5, 1.08)); ax.grid(axis='x', visible=False)
    R.fig(fig, 'per_class_auc', 'One-vs-rest AUC per ISUP grade on the pooled test set (dotted line = chance, 0.5).')

def f_calibration():
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, (k, cut, task) in zip(axes, [(0, 1, 'Cancer (ISUP ≥ 1)'), (1, 2, 'csPCa (ISUP ≥ 2)')]):
        for m in METHODS:
            t = sel(ENS[m][0], 'test', 'pooled'); yb = (t.isup_grade >= cut).values; p = t[f's{k}'].values
            edges = np.linspace(0, 1, 11); idx = np.clip(np.digitize(p, edges) - 1, 0, 9)
            xs = [p[idx == i].mean() for i in range(10) if (idx == i).sum() >= 5]; ys = [yb[idx == i].mean() for i in range(10) if (idx == i).sum() >= 5]
            ax.plot(xs, ys, color=METHOD_COLORS[m], ls=METHOD_LS[m], marker=METHOD_MARK[m], ms=5, label=f'{NICE[m]} (ECE {ece(yb, p):.3f})')
        ax.plot([0, 1], [0, 1], color=INK2, ls=':', lw=1); ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed frequency'); ax.set_title(task); ax.legend(fontsize=7)
    fig.suptitle('Reliability diagrams (pooled test set)', fontweight='bold', y=1.02)
    R.fig(fig, 'calibration_reliability', 'Reliability diagrams: predicted probability vs observed frequency (dotted = perfect calibration); ECE = expected calibration error.')

def f_score_dist():
    b, _ = ENS[best_fl]; t = sel(b, 'test', 'pooled')
    fig, axes = plt.subplots(1, len(HOSP), figsize=(12, 4), sharey=True)
    for ax, h in zip(axes, HOSP):
        d = t[t.hospital == h]; data = [d[d.isup_grade == g]['s1'].values for g in range(K)]
        bp = ax.boxplot(data, patch_artist=True, widths=.6, medianprops=dict(color=INK, lw=1.5), flierprops=dict(ms=3, markeredgecolor=INK2))
        for patch, c in zip(bp['boxes'], ISUP_COLORS): patch.set_facecolor(c); patch.set_edgecolor('white')
        thv = OPS.get((best_fl, 1, 'Youden (val)'))
        if thv is not None: ax.axhline(thv, color=INK2, ls=':', lw=1)
        ax.set_xticklabels([f'ISUP {g}' for g in range(K)], fontsize=8); ax.set_title(SITE_NICE[h])
    axes[0].set_ylabel('Predicted P(ISUP ≥ 2)')
    R.fig(fig, 'score_by_true_grade', f'Predicted probability of clinically significant cancer by true ISUP grade ({NICE[best_fl]}); dotted line = Youden threshold chosen on validation.')

def f_threshold():
    b, _ = ENS[best_fl]; t = sel(b, 'test', 'pooled'); yb = (t.isup_grade >= 2).values; p = t['s1'].values
    th = np.linspace(0, 1, 201)
    sens = [(p[yb] >= x).mean() for x in th]; spec = [(p[~yb] < x).mean() for x in th]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(th, sens, color='#2a78d6', label='Sensitivity'); ax.plot(th, spec, color='#eb6834', label='Specificity')
    for name, ls in [('Youden (val)', ':'), ('Sensitivity ≥ 90% (val)', '--'), ('Sensitivity ≥ 95% (val)', '-.')]:
        if (best_fl, 1, name) in OPS: ax.axvline(OPS[(best_fl, 1, name)], color=INK2, ls=ls, lw=1, label=name)
    ax.set_xlabel('Decision threshold on P(ISUP ≥ 2)'); ax.set_ylabel('Rate'); ax.set_title(f'Threshold analysis – {NICE[best_fl]} (test)'); ax.legend(fontsize=8)
    R.fig(fig, 'threshold_analysis_cspca', 'Sensitivity and specificity for csPCa across decision thresholds on the pooled test set; vertical lines = thresholds fixed on validation.')

def f_cross_site():
    fig, axes = plt.subplots(1, 3, figsize=(15, 0.6 * len(METHODS) + 2))
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list('b', ['#ffffff'] + ISUP_COLORS)
    for ax, (c, t_) in zip(axes, [('auc_cancer', 'AUC cancer'), ('auc_cspca', 'AUC csPCa'), ('qwk', 'QWK')]):
        M = np.array([[metrics_from_s(sel(ENS[m][0], 'test', h).isup_grade.values, sel(ENS[m][0], 'test', h)[SC].values, ENS[m][1])[c] for h in HOSP] for m in METHODS])
        ax.imshow(M, cmap=cmap, vmin=max(0, np.nanmin(M) - 0.1), vmax=1, aspect='auto')
        for i in range(len(METHODS)):
            for j in range(len(HOSP)):
                ax.text(j, i, f'{M[i, j]:.3f}', ha='center', va='center', fontsize=9, color='white' if M[i, j] > (np.nanmin(M) + 1) / 2 else INK)
        ax.set_xticks(range(len(HOSP))); ax.set_xticklabels([f'Test: {h.split("_")[-1]}' for h in HOSP]); ax.set_yticks(range(len(METHODS)))
        ax.set_yticklabels([NICE[m] for m in METHODS] if c == 'auc_cancer' else [''] * len(METHODS)); ax.set_title(t_); ax.grid(False)
    fig.suptitle('Cross-site generalisation matrix', fontweight='bold', y=1.03)
    R.fig(fig, 'cross_site_heatmap', 'Each model tested separately at each hospital: single-hospital models degrade on the other hospital, while federated models generalise to both.')

for fn, what in [(f_confusion_grid, 'confusion'), (f_per_class_auc, 'per-class AUC'), (f_calibration, 'calibration'),
                 (f_score_dist, 'score distribution'), (f_threshold, 'threshold analysis'), (f_cross_site, 'cross-site heatmap')]:
    R.safe(fn, what)

In [ ]:
# ── Cell 9: FIGURES – explainability (attention heatmaps + top-attention tiles) ─
import torch.nn as nn
class GatedABMIL(nn.Module):   # identical to NB02
    def __init__(self, d_in, hid=256, attn=128, drop=0.25, n_out=K - 1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d_in, hid), nn.GELU(), nn.Dropout(drop))
        self.att_a = nn.Sequential(nn.Linear(hid, attn), nn.Tanh()); self.att_b = nn.Sequential(nn.Linear(hid, attn), nn.Sigmoid())
        self.att_w = nn.Linear(attn, 1); self.head = nn.Sequential(nn.LayerNorm(hid), nn.Dropout(drop), nn.Linear(hid, n_out))
    def forward(self, x, mask):
        h = self.fc(x); s = self.att_w(self.att_a(h) * self.att_b(h)).squeeze(-1).masked_fill(~mask, -1e4)
        a = torch.softmax(s, 1); return self.head(torch.bmm(a.unsqueeze(1), h).squeeze(1)), a

def f_attention():
    import openslide
    PANDA = find_panda_dir(); assert PANDA, 'PANDA competition data not attached'
    FEAT_DIR, MAN = fetch_feature_store(CFG['backbone'], CFG['smoke'])
    hm = CFG['heatmap_method'] if CFG['heatmap_method'] in METHODS else best_fl
    ck = torch.load(OUT / f'model__{hm}__s{CFG["heatmap_seed"]}.pt', map_location='cpu', weights_only=False)
    net = GatedABMIL(ck['d_in'], TCFG['hid'], TCFG['attn'], TCFG['dropout']); net.load_state_dict(ck['state']); net.eval()
    te = sel(ENS[hm][0], 'test', 'pooled')
    pick = pd.concat([te[(te.hospital == h) & (te.isup_grade == g)].head(1) for h in HOSP for g in (0, 2, 5)]).head(CFG['n_heatmaps'])
    feats, coords = load_features(FEAT_DIR, MAN, ids=pick.image_id, with_coords=True)
    fig, axes = plt.subplots(len(pick), 2, figsize=(14, 3.2 * len(pick))); axes = np.atleast_2d(axes)
    fig2, axes2 = plt.subplots(len(pick), 8, figsize=(14, 1.9 * len(pick))); axes2 = np.atleast_2d(axes2)
    for row, row2, (_, r) in zip(axes, axes2, pick.iterrows()):
        f = feats[r.image_id].float()[None]
        with torch.no_grad(): lg, a = net(f, torch.ones(1, f.shape[1], dtype=torch.bool))
        a = a[0].numpy(); s_ = mono_sigmoid(lg.numpy())
        sl = openslide.OpenSlide(str(PANDA / 'train_images' / f'{r.image_id}.tiff')); lv = sl.level_count - 1; ds = sl.level_downsamples[lv]
        rgba = np.asarray(sl.read_region((0, 0), lv, sl.level_dimensions[lv])); th = rgba[..., :3].copy(); th[rgba[..., 3] == 0] = 255
        heat = np.zeros(th.shape[:2]); T = MAN['tile_px'] / ds; an = (a - a.min()) / (a.max() - a.min() + 1e-9)
        cc = coords[r.image_id].numpy()
        for (x, y), v in zip(cc, an): heat[int(y / ds):int(y / ds + T), int(x / ds):int(x / ds + T)] = v
        row[0].imshow(th); row[0].set_title(f'{SITE_NICE[r.hospital]} · true ISUP {r.isup_grade}', fontsize=9, fontweight='normal')
        row[1].imshow(th); row[1].imshow(np.ma.masked_where(heat == 0, heat), cmap=matplotlib.colors.LinearSegmentedColormap.from_list('b', ISUP_COLORS), alpha=.6)
        row[1].set_title(f'Attention · P(cancer) {s_[0,0]:.2f} · P(ISUP≥2) {s_[0,1]:.2f}', fontsize=9, fontweight='normal')
        for ax in row: ax.axis('off')
        top = np.argsort(-a)[:8]
        for j, ax in enumerate(row2):
            ax.axis('off')
            if j < len(top):
                x, y = cc[top[j]]; ax.imshow(np.asarray(sl.read_region((int(x), int(y)), 0, (MAN['tile_px'], MAN['tile_px'])))[..., :3])
        row2[0].set_title(f'ISUP {r.isup_grade} ({r.hospital.split("_")[-1]}) – top-8 attended tiles', fontsize=8, loc='left', fontweight='normal')
        sl.close()
    fig.suptitle(f'Attention heatmaps – {NICE[hm]}', fontweight='bold', y=1.0)
    R.fig(fig, 'attention_heatmaps', f'Whole-slide attention heatmaps of {NICE[hm]} on test slides (darker blue = higher attention) with predicted probabilities.')
    fig2.suptitle('Highest-attention tiles (20×) per slide', fontweight='bold', y=1.0)
    R.fig(fig2, 'top_attention_tiles', 'The eight tiles with the highest attention weight for each slide above – the regions driving the prediction, for pathologist review.')

R.safe(f_attention, 'attention heatmaps / top tiles')

In [ ]:
# ── Cell 10: Collect NB01/NB02 reports, master index, push to HF + private Kaggle ─
collected = []
for repo, tag in [(NAMES['features'], 'P'), (TRAIN_REPO, 'S')]:
    try:
        cb = src if repo == TRAIN_REPO else CloudBackup(repo, repo, repo, WORK / f'_rep_{tag}', enable_kaggle=False)
        for f in cb.list_remote():
            if f.startswith((f'fig_{tag}', f'tab_{tag}', f'REPORT_INDEX_{tag}')) and not (FIG / f).exists():
                if cb.pull(f, FIG / f): collected.append(f)
        for p in glob.glob(f'/kaggle/input/**/{repo}/*', recursive=True) + [str(q) for q in NAMES['feat_dir'].glob('*')] + [str(q) for q in OUT.glob('*')]:
            n = Path(p).name
            if n.startswith((f'fig_{tag}', f'tab_{tag}', f'REPORT_INDEX_{tag}')) and not (FIG / n).exists():
                shutil.copy(p, FIG / n); collected.append(n)
    except Exception as e:
        log(f'⚠️  could not collect {tag} report: {e}')
log(f'Collected {len(collected)} files from the preprocessing/split reports')

files = R.finalize()
parts = []
for tag, title in [('P', 'Preprocessing (NB01)'), ('S', 'Data split & feature space (NB02)'), ('E', 'Evaluation (NB03)')]:
    ip = FIG / f'REPORT_INDEX_{tag}.csv'
    if ip.exists():
        d = pd.read_csv(ip); parts.append(f'\n## {title}\n')
        parts += [f'- **{r.id}** — `{r.file}`: {r.caption}' for r in d.itertuples()]
nfig = len([p for p in FIG.glob('fig_*.png') if re.match(r'fig_[PSE]\d', p.name)]); ntab = len([p for p in FIG.glob('tab_*.csv') if re.match(r'tab_[PSE]\d', p.name)])
master = [f'# Thesis figures and tables – federated Gleason grading ({CFG["backbone"]})',
          f'Generated {datetime.now():%Y-%m-%d %H:%M} · {nfig} figures (PNG 300 dpi + PDF) · {ntab} tables (CSV + MD + LaTeX)',
          f'Best federated method (validation): **{NICE[best_fl]}**'] + parts
(FIG / 'THESIS_FIGURES_AND_TABLES.md').write_text('\n'.join(master) + '\n')
try:
    with pd.ExcelWriter(FIG / 'ALL_TABLES.xlsx') as xw:
        for c in sorted(p for p in FIG.glob('tab_*.csv') if re.match(r'tab_[PSE]\d', p.name)):
            pd.read_csv(c).to_excel(xw, sheet_name=c.stem[4:35], index=False)
except Exception as e:
    log(f'(xlsx workbook skipped: {e})')

all_files = sorted(p for p in FIG.iterdir() if p.is_file() and p.suffix != '.tmp' and p.name != 'dataset-metadata.json')
for i in range(0, len(all_files), 50):                      # HF: few large commits
    res_backup.push_many(all_files[i:i + 50], msg=f'thesis report part {i // 50 + 1}')
res_backup.kaggle_final_sync(f'final results: {nfig} figures, {ntab} tables')
log(f'🏁 NB03 complete: {nfig} figures and {ntab} tables in {FIG} (+ HF + private Kaggle). See THESIS_FIGURES_AND_TABLES.md')